# 09 — Topology localisation and ambiguity

This notebook audits the location attached to each incident. A location is a
ranked physical scope supported by the observed entity footprint—not a causal
root-cause claim. If two topology nodes have identical observable descendants,
the result is explicitly marked ambiguous rather than choosing one arbitrarily.
Geographic clusters remain context and cannot be reported as physical roots.


## 1. Setup


In [ ]:
from pathlib import Path
import os
import sys

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")


def find_repository(start=Path.cwd()):
    """Find the checked-out repository when Jupyter starts in any subfolder."""
    override = os.getenv("TELCO_PROJECT_ROOT")
    if override:
        candidates = [Path(override).expanduser().resolve()]
    else:
        start = start.resolve()
        candidates = [start, *start.parents]
        if "google.colab" in sys.modules:
            candidates += [
                Path("/content/drive/MyDrive/anomaly_detection"),
                Path("/content/drive/MyDrive/telco-anomaly-detection"),
            ]
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Open this notebook from the cloned repository, or set TELCO_PROJECT_ROOT."
    )


PROJECT_ROOT = find_repository()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import json

import pandas as pd
from IPython.display import display

from telco_anomaly.localisation import localisation_view, topology_identifiability
from telco_anomaly.io import (
    immutable_output_directory,
    read_json,
    resolve_data_root,
    write_json,
)

DATA_ROOT = resolve_data_root()
CORE_RUN_ID = os.getenv("PON_CORE_RUN_ID", "synthetic_pon_core_v1")
INCIDENT_RUN_ID = os.getenv("PON_INCIDENT_RUN_ID", "synthetic_pon_incidents_v1")
LOCALISATION_RUN_ID = os.getenv("PON_LOCALISATION_RUN_ID", "synthetic_pon_localisation_v1")

CORE_ROOT = DATA_ROOT / "core" / "synthetic_pon" / CORE_RUN_ID / "SPEC-CORE"
INCIDENT_ROOT = DATA_ROOT / "incidents" / "synthetic_pon" / INCIDENT_RUN_ID
OUTPUT_ROOT = DATA_ROOT / "localisation" / "synthetic_pon" / LOCALISATION_RUN_ID

cases = pd.read_parquet(INCIDENT_ROOT / "incidents.parquet")
members = pd.read_parquet(INCIDENT_ROOT / "incident_members.parquet")
topology = pd.read_parquet(CORE_ROOT / "topology_memberships.parquet")
physical = topology.loc[topology["group_family"].eq("physical_topology")].copy()

display(pd.Series({
    "incidents": len(cases),
    "physical_memberships": len(physical),
    "truth_opened": False,
}, name="value").to_frame())


## 2. Build observable topology footprints


In [ ]:
footprints = {
    (str(group_type), str(group_id)): frozenset(group["entity_id"].astype(str))
    for (group_type, group_id), group in physical.groupby(["group_type", "group_id"])
}
equivalents = {}
for key, footprint in footprints.items():
    equivalents[key] = sorted(
        candidate for candidate, value in footprints.items() if value == footprint
    )

identifiability = topology_identifiability(physical)
footprint_table = identifiability.rename(columns={
    "group_type": "scope_type",
    "group_id": "scope_id",
    "equivalence_size": "equivalent_scope_count",
})
footprint_table["descendant_entities"] = footprint_table["descendants"].map(len)
display(footprint_table.groupby("scope_type").agg(
    scopes=("scope_id", "nunique"),
    median_descendants=("descendant_entities", "median"),
    ambiguous_scopes=("equivalent_scope_count", lambda values: int((values > 1).sum())),
))


## 3. Make ambiguity explicit in the ranked incident table


In [ ]:
def localisation_record(row):
    if str(row.scope_type) == "entity":
        candidates = [("entity", str(row.scope_id))]
    else:
        candidates = equivalents.get((str(row.scope_type), str(row.scope_id)), [])
    return pd.Series({
        "location_candidates": json.dumps([
            {"scope_type": scope_type, "scope_id": scope_id}
            for scope_type, scope_id in candidates
        ]),
        "location_candidate_count": len(candidates),
        "location_ambiguous": len(candidates) > 1,
        "location_claim": (
            "entity evidence" if str(row.scope_type) == "entity"
            else "physical scope evidence"
        ),
    })


if len(cases):
    additions = cases.apply(localisation_record, axis=1)
else:
    additions = pd.DataFrame(columns=[
        "location_candidates", "location_candidate_count",
        "location_ambiguous", "location_claim",
    ])
localised = pd.concat([cases.reset_index(drop=True), additions], axis=1)
if localised["scope_type"].eq("geo_cluster").any():
    raise AssertionError("Geographic context cannot be emitted as a physical location")

display(localisation_view(cases).head(30))
display(localised[[
    "rank", "case_id", "case_start", "scope_type", "scope_id",
    "location_ambiguous", "affected_entity_count", "channels",
    "leading_features", "anomaly_evidence_score",
]].head(30))


## 4. Save localised incidents


In [ ]:
summary = pd.DataFrame([
    {"measure": "incidents", "value": len(localised)},
    {"measure": "entity_locations", "value": int(localised["scope_type"].eq("entity").sum())},
    {"measure": "infrastructure_locations", "value": int((~localised["scope_type"].eq("entity")).sum())},
    {"measure": "ambiguous_locations", "value": int(localised["location_ambiguous"].sum())},
])
localisation_manifest = {
    "dataset": "synthetic_pon",
    "partition": "development",
    "claim": "ranked physical scope, not causal root cause",
    "truth_files_read": [],
    "incidents": len(localised),
    "ambiguous_locations": int(localised["location_ambiguous"].sum()),
}

if OUTPUT_ROOT.exists():
    print("Using existing immutable localisation:", OUTPUT_ROOT)
else:
    with immutable_output_directory(OUTPUT_ROOT) as output:
        localised.to_parquet(output / "localised_incidents.parquet", index=False)
        summary.to_parquet(output / "localisation_summary.parquet", index=False)
        write_json(output / "localisation_manifest.json", localisation_manifest)

display(summary)
print("Next: 10_LOCKED_EVALUATION.ipynb")
